In [1]:
%pip install openpyxl
%pip install pandas
%pip install python-dotenv

  Using cached openpyxl-3.1.5-py2.py3-none-any.whl.metadata (2.5 kB)
  Using cached et_xmlfile-2.0.0-py3-none-any.whl.metadata (2.7 kB)
Using cached openpyxl-3.1.5-py2.py3-none-any.whl (250 kB)
Using cached et_xmlfile-2.0.0-py3-none-any.whl (18 kB)
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


  Using cached pandas-3.0.6-cp312-cp312-win_amd64.whl.metadata (19 kB)
  Using cached numpy-2.5.4-cp312-cp312-win_amd64.whl.metadata (6.6 kB)
Using cached pandas-3.0.6-cp312-cp312-win_amd64.whl (9.7 MB)
Using cached numpy-2.5.4-cp312-cp312-win_amd64.whl (12.6 MB)
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


  Using cached python_dotenv-1.2.4-py3-none-any.whl.metadata (29 kB)
Using cached python_dotenv-1.2.4-py3-none-any.whl (23 kB)
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
import os
import re

import numpy as np
import pandas as pd

# Sistema especialista para recomendação de jogos baseado em hardware e orçamento do usuário.
# Versão adaptada para o CSV da Steam (games.csv).

BASE_DIR = "C:\\Users\\<seu_usuario>\\Desktop\\IAGames\\API"  # altere para o caminho do seu PC

# Arquivos usados (dentro de BASE_DIR):
#   Jogos\games.csv      -> CSV da Steam
#   Hardware\index.csv   -> tabela de CPUs/GPUs (colunas "Nome" e "Tier (Desempenho)")
ARQUIVO_JOGOS = os.path.join(BASE_DIR, "Jogos", "games.csv")
ARQUIVO_HARDWARE = os.path.join(BASE_DIR, "Hardware", "index.csv")

# O CSV da Steam tem 40 campos por linha, mas o cabeçalho original vem com
# "Discount" e "DLC count" grudados ("DiscountDLC count"), ou seja, 39 nomes.
# Se o pandas lesse o cabeçalho como está, TODAS as colunas ficariam deslocadas.
# Por isso o cabeçalho é ignorado e estes nomes corretos são usados no lugar.
COLUNAS_STEAM = [
    "AppID", "Name", "Release date", "Estimated owners", "Peak CCU", "Required age",
    "Price", "Discount", "DLC count", "About the game", "Supported languages",
    "Full audio languages", "Reviews", "Header image", "Website", "Support url",
    "Support email", "Windows", "Mac", "Linux", "Metacritic score", "Metacritic url",
    "User score", "Positive", "Negative", "Score rank", "Achievements", "Recommendations",
    "Notes", "Average playtime forever", "Average playtime two weeks",
    "Median playtime forever", "Median playtime two weeks", "Developers", "Publishers",
    "Categories", "Genres", "Tags", "Screenshots", "Movies",
]

# Só as colunas que o sistema realmente usa (o arquivo é grande; assim economiza memória)
COLUNAS_USADAS = [
    "AppID", "Name", "Release date", "Price", "Windows",
    "Positive", "Negative", "Categories", "Genres", "Tags",
]

# Quantos jogos (os mais bem avaliados) são avaliados / exibidos na opção 1
LIMITE_AVALIACAO = 300
LIMITE_EXIBICAO = 15

# O CSV da Steam NÃO traz requisitos de CPU/GPU/RAM. Por isso o nível exigido por
# cada jogo é ESTIMADO a partir das tags/gêneros, ano de lançamento e preço.
TAGS_LEVES = [
    "pixel graphics", "2d", "retro", "text-based", "visual novel", "card game", "puzzle",
    "casual", "turn-based", "minimalist", "hand-drawn", "point & click", "idle", "clicker",
]
TAGS_GPU_PESADA = [
    "realistic", "photorealistic", "open world", "third person", "first-person", "fps",
    "3d", "cinematic", "survival horror",
]
TAGS_CPU_PESADA = [
    "strategy", "simulation", "city builder", "grand strategy", "4x", "rts",
    "massively multiplayer", "management", "base building", "colony sim", "automation",
    "building", "sandbox", "open world",
]

# Tradução simples de termos em português para as tags/gêneros em inglês da Steam
TRADUCAO_CATEGORIAS = {
    "terror": "horror", "estratégia": "strategy", "estrategia": "strategy",
    "corrida": "racing", "esportes": "sports", "esporte": "sports",
    "aventura": "adventure", "ação": "action", "acao": "action",
    "simulação": "simulation", "simulacao": "simulation", "plataforma": "platformer",
    "quebra-cabeça": "puzzle", "quebra-cabeca": "puzzle", "mundo aberto": "open world",
    "luta": "fighting", "tiro": "shooter", "pixel": "pixel graphics",
    "cooperativo": "co-op", "multijogador": "multiplayer", "sobrevivência": "survival",
    "sobrevivencia": "survival", "fazenda": "farming", "indie": "indie", "rpg": "rpg",
}


class Sistema:
    def menu(self):
        """Pergunta o que o usuário quer fazer e retorna a opção escolhida ('1' ou '2')."""
        print("O que você quer fazer?")
        print("1 - Saber os tipos de jogos disponíveis dentro do meu orçamento e que rodam no meu PC")
        print("2 - Saber se um jogo específico roda no meu PC e cabe no meu orçamento")

        opcao = input("Digite 1 ou 2: ").strip()

        while opcao not in ("1", "2"):
            opcao = input("Opção inválida. Digite 1 ou 2: ").strip()

        return opcao

    def __init__(self, jogo, preco, cpu, gpu, ram):
        self.jogo = jogo
        self.preco = preco
        self.cpu = cpu
        self.gpu = gpu
        self.ram = ram

        # dados de RAM extras coletados em Hardware(), usados pela base de regras
        self.geracao_ram = None
        self.mhz = None

        # guarda os dados encontrados no Hardware() para reusar na recomendação
        self.cpu_encontrada = None
        self.gpu_encontrada = None
        self.coluna_jogo = "Name"
        self.hardware_df = None

        # colunas de requisitos (estimados) criadas em carregar_jogos()
        self.coluna_cpu_req = "CPU Estimada"
        self.coluna_gpu_req = "GPU Estimada"
        self.coluna_ram_req = "RAM Estimada (GB)"

        # tabela de jogos da Steam (carregada sob demanda, uma única vez)
        self.jogos_df = None

        # -------------------------------------------------------------
        # BASE DE CONHECIMENTO (requisito 2): lista de regras nomeadas
        # -------------------------------------------------------------
        self.regras = self._criar_base_regras()

    # ------------------------------------------------------------------
    # Utilitários
    # ------------------------------------------------------------------

    def _mostrar(self, tabela):
        """Mostra uma tabela: usa display() no Jupyter e print() fora dele."""
        try:
            display(tabela)  # noqa: F821  (existe apenas no Jupyter/IPython)
        except NameError:
            print(tabela.to_string())

    # ------------------------------------------------------------------
    # Base de conhecimento + Motor de decisão (requisitos 2 e 3)
    # ------------------------------------------------------------------

    def _criar_base_regras(self):
        """
        Base de conhecimento do sistema: cada regra é um dicionário com
        - id / nome: identificação da regra
        - categoria: 'orcamento', 'hardware' ou 'positivo'
        - severidade: 'bloqueia' (impede rodar/comprar), 'alerta' (aviso,
          não impede) ou 'info' (observação positiva)
        - quando(fatos): condição SE
        - mensagem(fatos): explicação ENTÃO, usada na resposta ao usuário

        `fatos` é um dicionário com o estado do usuário + do jogo avaliado.
        """
        return [
            {
                "id": "R1",
                "nome": "RAM mínima geral",
                "categoria": "hardware",
                "severidade": "bloqueia",
                "quando": lambda f: f.get("ram") is not None and f["ram"] < 8,
                "mensagem": lambda f: (
                    f"Você tem {f['ram']}GB de RAM, abaixo do mínimo geral (8GB) "
                    "para jogos atuais."
                ),
            },
            {
                "id": "R2",
                "nome": "RAM abaixo do requisito do jogo",
                "categoria": "hardware",
                "severidade": "bloqueia",
                "quando": lambda f: (
                    f.get("ram") is not None
                    and f.get("ram_req") is not None
                    and not pd.isna(f["ram_req"])
                    and f["ram"] < f["ram_req"]
                ),
                "mensagem": lambda f: (
                    f"RAM insuficiente para este jogo: você tem {f['ram']}GB "
                    f"e o jogo pede {f['ram_req']}GB (estimado)."
                ),
            },
            {
                "id": "R3",
                "nome": "RAM DDR3 com frequência baixa",
                "categoria": "hardware",
                "severidade": "alerta",
                "quando": lambda f: (
                    (f.get("geracao_ram") or "").upper() == "DDR3"
                    and f.get("mhz") is not None
                    and f["mhz"] < 1600
                ),
                "mensagem": lambda f: (
                    f"Memória DDR3 a {f['mhz']}MHz está abaixo do recomendado "
                    "(1600MHz), pode prejudicar o desempenho."
                ),
            },
            {
                "id": "R4",
                "nome": "RAM DDR4 com frequência baixa",
                "categoria": "hardware",
                "severidade": "alerta",
                "quando": lambda f: (
                    (f.get("geracao_ram") or "").upper() == "DDR4"
                    and f.get("mhz") is not None
                    and f["mhz"] < 2400
                ),
                "mensagem": lambda f: (
                    f"Memória DDR4 a {f['mhz']}MHz está abaixo do recomendado "
                    "(2400MHz), pode prejudicar o desempenho."
                ),
            },
            {
                "id": "R5",
                "nome": "RAM DDR5 com frequência baixa",
                "categoria": "hardware",
                "severidade": "alerta",
                "quando": lambda f: (
                    (f.get("geracao_ram") or "").upper() == "DDR5"
                    and f.get("mhz") is not None
                    and f["mhz"] < 3200
                ),
                "mensagem": lambda f: (
                    f"Memória DDR5 a {f['mhz']}MHz está abaixo do recomendado "
                    "(3200MHz), pode prejudicar o desempenho."
                ),
            },
            {
                "id": "R6",
                "nome": "Orçamento insuficiente",
                "categoria": "orcamento",
                "severidade": "bloqueia",
                "quando": lambda f: (
                    f.get("valor_jogo") is not None
                    and f.get("preco") is not None
                    and f["valor_jogo"] > f["preco"]
                ),
                "mensagem": lambda f: (
                    f"O jogo custa US$ {f['valor_jogo']:.2f} e seu orçamento é US$ {f['preco']:.2f}."
                ),
            },
            {
                "id": "R7",
                "nome": "CPU abaixo do requisito mínimo",
                "categoria": "hardware",
                "severidade": "bloqueia",
                "quando": lambda f: f.get("cpu_ok") is False,
                "mensagem": lambda f: (
                    "Sua CPU está abaixo do requisito mínimo estimado do jogo "
                    f"(classificação: {f.get('cpu_classificacao')})."
                ),
            },
            {
                "id": "R8",
                "nome": "GPU abaixo do requisito mínimo",
                "categoria": "hardware",
                "severidade": "bloqueia",
                "quando": lambda f: f.get("gpu_ok") is False,
                "mensagem": lambda f: (
                    "Sua GPU está abaixo do requisito mínimo estimado do jogo "
                    f"(classificação: {f.get('gpu_classificacao')})."
                ),
            },
            {
                "id": "R9",
                "nome": "CPU não avaliável numericamente",
                "categoria": "hardware",
                "severidade": "alerta",
                "quando": lambda f: (
                    f.get("cpu_ok") is None
                    and f.get("cpu_classificacao")
                    and "Não avaliado" in str(f["cpu_classificacao"])
                ),
                "mensagem": lambda f: (
                    f"Não foi possível comparar sua CPU numericamente: "
                    f"{f['cpu_classificacao']}."
                ),
            },
            {
                "id": "R10",
                "nome": "GPU não avaliável numericamente",
                "categoria": "hardware",
                "severidade": "alerta",
                "quando": lambda f: (
                    f.get("gpu_ok") is None
                    and f.get("gpu_classificacao")
                    and "Não avaliado" in str(f["gpu_classificacao"])
                ),
                "mensagem": lambda f: (
                    f"Não foi possível comparar sua GPU numericamente: "
                    f"{f['gpu_classificacao']}."
                ),
            },
            {
                "id": "R11",
                "nome": "Desempenho excelente",
                "categoria": "positivo",
                "severidade": "info",
                "quando": lambda f: (
                    f.get("cpu_classificacao") == "Roda muito bem"
                    and f.get("gpu_classificacao") == "Roda muito bem"
                ),
                "mensagem": lambda f: (
                    "Sua CPU e GPU estão bem acima do requisito mínimo — "
                    "o jogo deve rodar muito bem."
                ),
            },
            {
                "id": "R12",
                "nome": "RAM exatamente no limite mínimo",
                "categoria": "hardware",
                "severidade": "alerta",
                "quando": lambda f: (
                    f.get("ram") is not None
                    and f.get("ram_req") is not None
                    and not pd.isna(f["ram_req"])
                    and f["ram"] == f["ram_req"]
                ),
                "mensagem": lambda f: (
                    "Sua RAM está exatamente no requisito mínimo estimado do jogo — "
                    "pode ser justo em momentos de pico de uso."
                ),
            },
        ]

    def mostrar_base_conhecimento(self):
        """Lista as regras da base de conhecimento (transparência do sistema)."""
        print(f"\n📚 Base de conhecimento carregada: {len(self.regras)} regras")
        for r in self.regras:
            print(f"  [{r['id']}] {r['nome']}  (categoria: {r['categoria']}, severidade: {r['severidade']})")
        print()

    def _motor_inferencia(self, fatos):
        """
        MOTOR DE DECISÃO (requisito 3): percorre a base de regras e retorna
        a lista das regras que dispararam para os fatos fornecidos.
        Cada fato ausente é tolerado (a regra simplesmente não dispara).
        """
        disparadas = []
        for regra in self.regras:
            try:
                if regra["quando"](fatos):
                    disparadas.append({
                        "id": regra["id"],
                        "nome": regra["nome"],
                        "categoria": regra["categoria"],
                        "severidade": regra["severidade"],
                        "mensagem": regra["mensagem"](fatos),
                    })
            except Exception:
                # fato necessário para essa regra não está disponível — ignora
                continue
        return disparadas

    def _explicar(self, avaliacao):
        """
        EXPLICAÇÃO (requisito 5): imprime, regra a regra, por que o sistema
        chegou naquela decisão.
        """
        print("\nPor que essa decisão foi tomada?")
        if not avaliacao["explicacoes"]:
            print("  Nenhuma regra da base de conhecimento foi disparada — tudo dentro do esperado.")
            return

        tags = {"bloqueia": "🚫", "alerta": "⚠️ ", "info": "✅"}
        for regra in avaliacao["explicacoes"]:
            tag = tags.get(regra["severidade"], "•")
            print(f"  {tag} [{regra['id']} - {regra['nome']}] {regra['mensagem']}")

    # ------------------------------------------------------------------
    # Carga e preparo do CSV da Steam
    # ------------------------------------------------------------------

    def carregar_jogos(self):
        """
        Lê o CSV da Steam (uma única vez), corrige o cabeçalho deslocado, mantém
        só jogos de Windows com nome e preço válidos e ESTIMA o nível de CPU/GPU/RAM
        exigido por cada jogo (o CSV não tem esses requisitos).
        """
        if self.jogos_df is not None:
            return self.jogos_df

        print("Carregando a base da Steam (pode levar alguns segundos)...")
        dados = pd.read_csv(
            ARQUIVO_JOGOS,
            header=0,               # descarta o cabeçalho com defeito...
            names=COLUNAS_STEAM,    # ...e usa os 40 nomes corretos
            usecols=COLUNAS_USADAS,
            low_memory=False,
        )

        dados["Price"] = pd.to_numeric(dados["Price"], errors="coerce")
        dados = dados.dropna(subset=["Name", "Price"])
        dados = dados[dados["Windows"].astype(str).str.lower() == "true"].copy()
        dados["Positive"] = pd.to_numeric(dados["Positive"], errors="coerce").fillna(0)

        # texto único (gêneros + tags + categorias) usado nas buscas e nas estimativas
        dados["_texto"] = (
            dados["Genres"].fillna("") + "," + dados["Tags"].fillna("") + "," + dados["Categories"].fillna("")
        ).str.lower()

        self._estimar_requisitos(dados)
        self.jogos_df = dados.reset_index(drop=True)
        print(f"Base carregada: {len(self.jogos_df)} jogos para Windows.\n")
        return self.jogos_df

    def _estimar_requisitos(self, dados):
        """
        Estima o nível exigido (Fraca / Média / Forte) de CPU e GPU e a RAM (GB)
        de cada jogo, a partir de tags/gêneros, ano de lançamento e preço.
        É uma aproximação: o CSV da Steam não traz os requisitos reais.
        """
        def tem(padroes):
            return dados["_texto"].str.contains("|".join(re.escape(p) for p in padroes), regex=True)

        leve = tem(TAGS_LEVES)
        gpu_pesada = tem(TAGS_GPU_PESADA)
        cpu_pesada = tem(TAGS_CPU_PESADA)

        ano = dados["Release date"].astype(str).str.extract(r"(\d{4})")[0].astype(float).fillna(2015)
        preco = dados["Price"].fillna(0)
        parece_aaa = (ano >= 2018) & (preco >= 40)

        gpu_pts = 1 + (gpu_pesada & (ano >= 2012)).astype(int) + parece_aaa.astype(int)
        gpu_pts = gpu_pts.where(~(leve & ~gpu_pesada), 1)   # jogo leve e sem tag pesada -> fraca
        gpu_pts = gpu_pts.where(ano >= 2012, 1)             # jogos antigos -> fraca
        gpu_pts = gpu_pts.clip(upper=3)

        cpu_pts = 1 + cpu_pesada.astype(int) + parece_aaa.astype(int)
        cpu_pts = cpu_pts.where(ano >= 2012, 1)
        cpu_pts = cpu_pts.clip(upper=3)

        niveis = {1: "Fraca", 2: "Média", 3: "Forte"}
        dados[self.coluna_gpu_req] = gpu_pts.map(niveis)
        dados[self.coluna_cpu_req] = cpu_pts.map(niveis)
        dados[self.coluna_ram_req] = np.maximum(gpu_pts, cpu_pts).map({1: 4, 2: 8, 3: 16})

    def _detectar_coluna(self, dados, possiveis_nomes):
        """Procura, entre uma lista de nomes prováveis, qual existe no DataFrame."""
        return next((c for c in possiveis_nomes if c in dados.columns), None)

    def _coluna_score(self):
        """Tenta achar, na tabela de hardware, a coluna usada como pontuação/nível/tier."""
        if self.hardware_df is None:
            return None
        return self._detectar_coluna(
            self.hardware_df,
            [
                "Tier (Desempenho)", "Tier(Desempenho)", "Tier", "Nível (Desempenho)",
                "Pontuação", "Pontuacao", "Nível", "Nivel", "Score", "Benchmark", "Desempenho"
            ]
        )

    # Escala usada para converter níveis categóricos (Fraca/Média/Forte...) em
    # números comparáveis, quando a planilha não usa uma pontuação numérica.
    TIER_ORDEM = {
        "fraca": 1, "fraco": 1, "baixa": 1, "baixo": 1, "ruim": 1,
        "média": 2, "media": 2, "médio": 2, "medio": 2,
        "moderada": 2, "moderado": 2, "intermediária": 2, "intermediaria": 2,
        "razoável": 2, "razoavel": 2,
        "forte": 3, "alta": 3, "alto": 3, "boa": 3, "bom": 3,
        "muito forte": 4, "excelente": 4, "top": 4,
    }

    def _valor_ordinal(self, valor):
        """Converte um valor numérico OU categórico (Fraca/Média/Forte...) numa escala comparável."""
        if valor is None or (isinstance(valor, float) and pd.isna(valor)):
            return None
        if isinstance(valor, (int, float, np.integer, np.floating)):
            return float(valor)
        return self.TIER_ORDEM.get(str(valor).strip().lower())

    def _classificar_desempenho(self, score_usuario, score_requisito):
        """
        Compara score do usuário com o requisito mínimo e devolve um rótulo:
        Não roda / Roda muito mal / Roda mal / Roda mediano / Roda bem / Roda muito bem
        """
        if not score_requisito:
            return "Não foi possível avaliar"

        razao = score_usuario / score_requisito

        if razao < 0.5:
            return "Não roda"
        elif razao < 0.8:
            return "Roda muito mal"
        elif razao < 1.0:
            return "Roda mal"
        elif razao < 1.3:
            return "Roda mediano"
        elif razao < 1.8:
            return "Roda bem"
        else:
            return "Roda muito bem"

    def _comparar_score(self, requisito_nome, peca_encontrada, coluna_score):
        """
        Compara a peça do usuário (CPU ou GPU) com o nível/pontuação exigido
        pelo jogo. Funciona tanto se a exigência for um número (score/pontuação)
        quanto se for um nível categórico (ex: "Fraca", "Média", "Forte"),
        já que ambos passam por _valor_ordinal antes de comparar.
        Retorna (ok, score_usuario, score_requisito, classificacao).
        ok pode ser True/False/None (None = não deu pra avaliar).
        """
        if peca_encontrada is None:
            return None, None, None, "Não avaliado (hardware do usuário não identificado)"

        requisito_vazio = (
            requisito_nome is None
            or (isinstance(requisito_nome, float) and pd.isna(requisito_nome))
            or str(requisito_nome).strip() == ""
        )
        if requisito_vazio:
            return None, None, None, "Não avaliado (não foi possível estimar o requisito deste jogo)"

        if coluna_score is None or self.hardware_df is None:
            return None, None, None, "Não avaliado (index.csv não tem coluna de nível/pontuação, ex: 'Tier (Desempenho)')"

        valor_usuario_bruto = peca_encontrada[coluna_score]
        score_usuario = self._valor_ordinal(valor_usuario_bruto)
        score_requisito = self._valor_ordinal(requisito_nome)

        if score_usuario is None or score_requisito is None:
            return None, None, None, (
                f"Não avaliado (não reconheci o nível '{requisito_nome}' do jogo "
                f"ou '{valor_usuario_bruto}' do seu hardware)"
            )

        ok = score_usuario >= score_requisito
        classificacao = self._classificar_desempenho(score_usuario, score_requisito)
        return ok, score_usuario, score_requisito, classificacao

    def _avaliar_jogo(self, jogo_info, coluna_jogo):
        """
        Avalia UM jogo (uma linha da base da Steam) contra o preço e o hardware
        já capturados do usuário, usando a base de regras + o motor de
        inferência (requisitos 2, 3, 4 e 5 do desafio).
        """
        nome = jogo_info.get(coluna_jogo, "Jogo desconhecido")
        valor = jogo_info.get("Price")
        cpu_req = jogo_info.get(self.coluna_cpu_req)
        gpu_req = jogo_info.get(self.coluna_gpu_req)
        ram_req = jogo_info.get(self.coluna_ram_req)

        coluna_score = self._coluna_score()
        cpu_ok, _, _, cpu_class = self._comparar_score(cpu_req, self.cpu_encontrada, coluna_score)
        gpu_ok, _, _, gpu_class = self._comparar_score(gpu_req, self.gpu_encontrada, coluna_score)

        # --- FATOS: tudo que o motor de inferência precisa saber ---
        fatos = {
            "ram": self.ram,
            "ram_req": ram_req,
            "geracao_ram": self.geracao_ram,
            "mhz": self.mhz,
            "preco": self.preco,
            "valor_jogo": valor,
            "cpu_ok": cpu_ok,
            "gpu_ok": gpu_ok,
            "cpu_classificacao": cpu_class,
            "gpu_classificacao": gpu_class,
        }

        # --- MOTOR: percorre a base de regras contra os fatos ---
        disparadas = self._motor_inferencia(fatos)

        orcamento_ok = not any(
            r["severidade"] == "bloqueia" and r["categoria"] == "orcamento" for r in disparadas
        )
        hardware_ok = not any(
            r["severidade"] == "bloqueia" and r["categoria"] == "hardware" for r in disparadas
        )

        if not orcamento_ok and not hardware_ok:
            status = "Fora do orçamento e PC não suporta"
        elif not orcamento_ok:
            status = "Fora do orçamento"
        elif not hardware_ok:
            status = "PC não suporta"
        else:
            status = "Roda"

        motivos_bloqueio = [r["mensagem"] for r in disparadas if r["severidade"] == "bloqueia"]

        return {
            "nome": nome,
            "valor": valor,
            "orcamento_ok": orcamento_ok,
            "hardware_ok": hardware_ok,
            "status": status,
            "motivos": motivos_bloqueio,     # só os bloqueios, pra listas resumidas
            "explicacoes": disparadas,       # todas as regras disparadas, pro "por quê"
            "cpu_classificacao": cpu_class,
            "gpu_classificacao": gpu_class,
        }

    # ------------------------------------------------------------------
    # Opção 1: tipos de jogos que cabem no orçamento e rodam no PC
    # ------------------------------------------------------------------

    def _termos_busca(self, texto):
        """Devolve o termo digitado + sua tradução (se existir) para buscar nos gêneros/tags."""
        texto = texto.strip().lower()
        termos = [texto]
        if texto in TRADUCAO_CATEGORIAS:
            termos.append(TRADUCAO_CATEGORIAS[texto])
        return [t for t in termos if t]

    def procurar(self):
        try:
            self.jogo = str(input('Digite o tipo de jogo (gênero/tag, ex: terror, estratégia, RPG, indie): '))
            self.preco = float(input('Até quanto você pode gastar (em US$)? '))

            dados = self.carregar_jogos()

            padrao = "|".join(re.escape(t) for t in self._termos_busca(self.jogo))
            resultado = dados[dados["_texto"].str.contains(padrao, regex=True, na=False)]

            if resultado.empty:
                print("Nenhum jogo encontrado nessa categoria!")
                return

            # garante que já temos o hardware do usuário; se não tiver, pergunta agora
            if self.cpu_encontrada is None or self.gpu_encontrada is None or self.hardware_df is None:
                print("\nPreciso saber o hardware do seu PC para avaliar quais jogos rodam.")
                self.Hardware()

            # a Steam tem milhares de jogos por categoria: avalia só os mais bem avaliados
            total = len(resultado)
            resultado = resultado.sort_values("Positive", ascending=False).head(LIMITE_AVALIACAO)

            print(f"\n--- {total} jogos na categoria '{self.jogo}'; avaliando os {len(resultado)} mais bem avaliados ---\n")

            avaliacoes = [self._avaliar_jogo(j, self.coluna_jogo) for _, j in resultado.iterrows()]

            rodam = [a for a in avaliacoes if a["status"] == "Roda"]
            fora_orcamento = [a for a in avaliacoes if not a["orcamento_ok"]]
            nao_suportado = [a for a in avaliacoes if not a["hardware_ok"]]

            print(f"Top {min(LIMITE_EXIBICAO, len(rodam))} que rodam no seu PC e cabem no orçamento:\n")
            for avaliacao in rodam[:LIMITE_EXIBICAO]:
                print(f"- {avaliacao['nome']} (US$ {avaliacao['valor']:.2f}) -> {avaliacao['status']}")
                self._explicar(avaliacao)
                print()

            def nomes(lista, n=5):
                if not lista:
                    return "nenhum"
                extra = f" e mais {len(lista) - n}" if len(lista) > n else ""
                return ", ".join(a["nome"] for a in lista[:n]) + extra

            print("--- Resumo ---")
            print(f"✅ Rodam no seu PC e cabem no orçamento ({len(rodam)}): {nomes(rodam)}")
            print(f"💰 Fora do orçamento ({len(fora_orcamento)}): {nomes(fora_orcamento)}")
            print(f"🚫 PC não suporta ({len(nao_suportado)}): {nomes(nao_suportado)}")
            print("\nObs.: os requisitos de CPU/GPU/RAM dos jogos são ESTIMADOS (a base da Steam não os traz).")

        except Exception as erro:
            print(f"Ocorreu algum erro: {erro}")

    # ------------------------------------------------------------------
    # Hardware do usuário
    # ------------------------------------------------------------------

    def Hardware(self):
        try:
            self.cpu = str(input('Qual é o processador do seu computador ? '))
            print("""
                Para saber o seu processador acesse:

                Configuraçoes --> Sistema --> Sobre
            """)

            self.gpu = str(input('Qual é a placa de vídeo do seu computador ? '))
            print("""
                Para saber a sua placa de vídeo acesse:

                Ctrl + shift + esc --> Gerenciador de Tarefas --> Desempenho --> Gpu
            """)

            hardware = pd.read_csv(ARQUIVO_HARDWARE, sep=None, engine="python")
            self.hardware_df = hardware  # guarda a tabela inteira pra reusar na recomendação

            resultado_cpu = hardware[
                hardware['Nome'].str.contains(self.cpu, case=False, na=False, regex=False)
            ]
            cpu_df = resultado_cpu

            resultado_gpu = hardware[
                hardware['Nome'].str.contains(self.gpu, case=False, na=False, regex=False)
            ]
            gpu_df = resultado_gpu

            if resultado_cpu.empty:
                print('Processador não encontrado')
            else:
                self._mostrar(cpu_df)
                # guarda a primeira linha encontrada para usar na recomendação
                self.cpu_encontrada = cpu_df.iloc[0]

            if resultado_gpu.empty:
                print('Placa de vídeo não encontrada')
            else:
                self._mostrar(gpu_df)
                self.gpu_encontrada = gpu_df.iloc[0]

            self.ram = int(input('Qual é a quantidade de memória RAM do seu computador ? '))
            self.geracao_ram = str(input('Qual é a geração da sua memória RAM ? (DDR3, DDR4, DDR5) ')).upper()
            self.mhz = int(input('Qual é a frequência da sua memória RAM ? (em MHz) '))

            fatos_ram = {
                "ram": self.ram,
                "ram_req": None,
                "geracao_ram": self.geracao_ram,
                "mhz": self.mhz,
                "preco": None,
                "valor_jogo": None,
                "cpu_ok": None,
                "gpu_ok": None,
                "cpu_classificacao": None,
                "gpu_classificacao": None,
            }
            disparadas_ram = self._motor_inferencia(fatos_ram)
            if disparadas_ram:
                print("\nObservações sobre sua memória RAM:")
                for r in disparadas_ram:
                    print(f"  - {r['mensagem']}")
            else:
                print("\nSua quantidade e frequência de RAM são suficientes para rodar jogos atuais.")

        except Exception as erro:
            print(f"Ocorreu algum erro: {erro}")

    # ------------------------------------------------------------------
    # Opção 2: um jogo específico
    # ------------------------------------------------------------------

    def perguntar_jogo(self):
        """Só pergunta o nome do jogo e devolve a resposta (sem mexer na base)."""
        return str(input('Qual jogo você quer verificar ? (nome exato ou parcial) '))

    def pesquisar_jogo(self, nome_jogo):
        """
        Pesquisa APENAS o jogo pelo nome na base da Steam (sem categoria,
        sem preço, sem checar hardware). Retorna até 5 resultados, os mais
        bem avaliados primeiro.
        """
        try:
            dados = self.carregar_jogos()
            resultado = dados[
                dados[self.coluna_jogo].str.contains(nome_jogo, case=False, na=False, regex=False)
            ]
            return resultado.sort_values("Positive", ascending=False).head(5)

        except Exception as erro:
            print(f"Ocorreu algum erro: {erro}")
            return None

    def recomendar_jogo(self):
        """
        Usa perguntar_jogo() + pesquisar_jogo() para achar o jogo e,
        em seguida, roda o motor de inferência sobre a base de regras para
        decidir se o PC atende e se cabe no orçamento — explicando o porquê.
        """
        try:
            nome_jogo = self.perguntar_jogo()
            self.preco = float(input('Até quanto você pode gastar nesse jogo (em US$) ? '))

            resultado = self.pesquisar_jogo(nome_jogo)

            if resultado is None or resultado.empty:
                print("Jogo não encontrado na base de dados!")
                return

            # vários jogos com nome parecido: deixa o usuário escolher
            escolha = 0
            if len(resultado) > 1:
                print("\nEncontrei mais de um jogo:")
                for i, (_, j) in enumerate(resultado.iterrows(), start=1):
                    print(f"  {i} - {j[self.coluna_jogo]} (US$ {j['Price']:.2f})")
                texto = input("Qual deles? (número, Enter = 1) ").strip()
                if texto.isdigit() and 1 <= int(texto) <= len(resultado):
                    escolha = int(texto) - 1

            jogo_info = resultado.iloc[escolha]
            print(f"\nJogo encontrado: {jogo_info[self.coluna_jogo]}")
            self._mostrar(
                jogo_info[["Name", "Release date", "Price", "Genres",
                           self.coluna_cpu_req, self.coluna_gpu_req, self.coluna_ram_req]].to_frame().T
            )

            if self.cpu_encontrada is None or self.gpu_encontrada is None:
                print("\nNão consegui identificar seu processador/placa de vídeo na tabela de hardware.")
                return

            avaliacao = self._avaliar_jogo(jogo_info, self.coluna_jogo)

            if avaliacao["cpu_classificacao"]:
                print(f"\nResultado de CPU: {avaliacao['cpu_classificacao']}")
            if avaliacao["gpu_classificacao"]:
                print(f"Resultado de GPU: {avaliacao['gpu_classificacao']}")

            print(f"\n>>> Decisão final: {avaliacao['status']} <<<")

            self._explicar(avaliacao)
            print("\nObs.: os requisitos de CPU/GPU/RAM do jogo são ESTIMADOS (a base da Steam não os traz).")

        except Exception as erro:
            print(f"Ocorreu algum erro: {erro}")


if __name__ == "__main__":
    sistema1 = Sistema('', '', '', '', '')
    sistema1.mostrar_base_conhecimento()

    opcao = sistema1.menu()

    if opcao == '1':
        sistema1.procurar()
    elif opcao == '2':
        sistema1.Hardware()
        sistema1.recomendar_jogo()


📚 Base de conhecimento carregada: 12 regras
  [R1] RAM mínima geral  (categoria: hardware, severidade: bloqueia)
  [R2] RAM abaixo do requisito do jogo  (categoria: hardware, severidade: bloqueia)
  [R3] RAM DDR3 com frequência baixa  (categoria: hardware, severidade: alerta)
  [R4] RAM DDR4 com frequência baixa  (categoria: hardware, severidade: alerta)
  [R5] RAM DDR5 com frequência baixa  (categoria: hardware, severidade: alerta)
  [R6] Orçamento insuficiente  (categoria: orcamento, severidade: bloqueia)
  [R7] CPU abaixo do requisito mínimo  (categoria: hardware, severidade: bloqueia)
  [R8] GPU abaixo do requisito mínimo  (categoria: hardware, severidade: bloqueia)
  [R9] CPU não avaliável numericamente  (categoria: hardware, severidade: alerta)
  [R10] GPU não avaliável numericamente  (categoria: hardware, severidade: alerta)
  [R11] Desempenho excelente  (categoria: positivo, severidade: info)
  [R12] RAM exatamente no limite mínimo  (categoria: hardware, severidade: alerta)

O

,Nome,Tipo,Fabricante,Tier (Desempenho)
563,AMD Radeon RX 550,GPU,AMD,Fraca
573,AMD Radeon RX 5500 XT,GPU,AMD,Média



Sua quantidade e frequência de RAM são suficientes para rodar jogos atuais.

--- 24732 jogos na categoria 'rpg'; avaliando os 300 mais bem avaliados ---

Top 15 que rodam no seu PC e cabem no orçamento:

- Dota 2 (US$ 0.00) -> Roda

Por que essa decisão foi tomada?
  ⚠️  [R9 - CPU não avaliável numericamente] Não foi possível comparar sua CPU numericamente: Não avaliado (hardware do usuário não identificado).

- Terraria (US$ 4.99) -> Roda

Por que essa decisão foi tomada?
  ⚠️  [R9 - CPU não avaliável numericamente] Não foi possível comparar sua CPU numericamente: Não avaliado (hardware do usuário não identificado).

- The Elder Scrolls V: Skyrim (US$ 19.99) -> Roda

Por que essa decisão foi tomada?
  ⚠️  [R9 - CPU não avaliável numericamente] Não foi possível comparar sua CPU numericamente: Não avaliado (hardware do usuário não identificado).

- Hades (US$ 7.49) -> Roda

Por que essa decisão foi tomada?
  ⚠️  [R9 - CPU não avaliável numericamente] Não foi possível comparar sua CPU n